# Empirical Analysis: Law & Crime Topic Prevalence Under Cross-National and Document-Type Shift

This notebook reproduces the ACS analysis structure for an LLM-based measurement task.
We use an LLM (Llama 3.1 8B) to classify parliamentary and media texts from the
Comparative Agendas Project (CAP) as related to Law & Crime (CAP major topic code 12),
calibrate with Isotonic Regression and MCGrad on a balanced multi-country sample, then
measure prevalence estimation bias across a shift gradient from no shift to maximum
shift (unseen country + unseen document type).

**Sub-populations (7 total):**

| Country | Doc type | Language | N (approx) | Role |
|---------|----------|----------|------------|------|
| Denmark | Parl. questions | Danish | 110K | Calibration + test |
| Spain | Oral questions | Spanish | 43K | Calibration + test |
| Spain | Media -- El Pais | Spanish | 57K | OOD target |
| Spain | Media -- El Mundo | Spanish | 51K | OOD target |
| US | Congressional bills | English | 468K | Calibration + test |
| Belgium | TV news | Dutch | 136K | OOD target |
| Belgium | Newspaper | Dutch | 21K | OOD target |

**Calibration design:** Balanced sample from 3 sub-populations (Denmark questions,
Spain questions, US bills) gives MCGrad variation in doc_type, country, language,
decade, and party. OOD targets (Spain media, Belgium) are completely unseen during
calibration.

In [ ]:
import sys
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# MCGrad for multicalibration
from mcgrad import methods as mcgrad_methods

# Shared color palette
sys.path.insert(0, "..")
from plot_config import METHOD_COLORS

os.makedirs('../paper/images', exist_ok=True)

# Plotting defaults
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 8,
    'figure.dpi': 150,
})

## 1. Data Loading

Load CAP data for 7 sub-populations across 4 countries. Each CSV has columns
including `id`, `country`, `doc_type`, `party`, `year`, `majortopic`.

LLM scores are joined separately in the next step (after the inference pipeline
completes).

In [ ]:
# --- Data paths ---
# CSV files with CAP data. The doc_type and country columns in these files
# use the following values:
#   country: 'Denmark', 'Spain', 'United States', 'Belgium'
#   doc_type: 'parliamentary_question', 'media', 'bill', 'tv_news', 'newspaper'
# We overwrite these with notebook-canonical values for consistency.

DATA_DIR = os.path.join('data')

DATA_PATHS = {
    'denmark_questions':   os.path.join(DATA_DIR, 'denmark_questions_final.csv'),
    'spain_questions':     os.path.join(DATA_DIR, 'spain_questions_final.csv'),
    'spain_elpais':        os.path.join(DATA_DIR, 'spain_media_elpais_final.csv'),
    'spain_elmundo':       os.path.join(DATA_DIR, 'spain_media_elmundo_final.csv'),
    'us_bills':            os.path.join(DATA_DIR, 'us_bills_final.csv'),
    'belgium_tv':          os.path.join(DATA_DIR, 'belgium_tv_final.csv'),
    'belgium_newspaper':   os.path.join(DATA_DIR, 'belgium_newspaper_final.csv'),
}

# Canonical metadata values used throughout the notebook.
# These are written into each DataFrame to ensure consistency regardless
# of what the raw CSV files contain.
SUBPOP_META = {
    'denmark_questions':  {'country': 'Denmark',       'doc_type': 'parliamentary_question', 'language': 'Danish'},
    'spain_questions':    {'country': 'Spain',         'doc_type': 'parliamentary_question', 'language': 'Spanish'},
    'spain_elpais':       {'country': 'Spain',         'doc_type': 'media',                  'language': 'Spanish'},
    'spain_elmundo':      {'country': 'Spain',         'doc_type': 'media',                  'language': 'Spanish'},
    'us_bills':           {'country': 'United States', 'doc_type': 'bill',                   'language': 'English'},
    'belgium_tv':         {'country': 'Belgium',       'doc_type': 'tv_news',                'language': 'Dutch'},
    'belgium_newspaper':  {'country': 'Belgium',       'doc_type': 'newspaper',              'language': 'Dutch'},
}

LABEL_COLUMN = 'law_crime'    # binary: 1 if majortopic == 12
SCORE_COLUMN = 'llm_score'    # continuous LLM probability (joined in next cell)


def load_cap_subpop(key):
    """Load one CAP sub-population CSV and create binary law_crime label."""
    path = DATA_PATHS[key]
    meta = SUBPOP_META[key]
    df = pd.read_csv(path)
    # Create binary label from CAP major topic code 12
    df[LABEL_COLUMN] = (df['majortopic'] == 12).astype(int)
    # Overwrite metadata columns with canonical values
    df['country'] = meta['country']
    df['doc_type'] = meta['doc_type']
    df['subpop'] = key
    print(f"  {key}: {len(df):,} docs, "
          f"law_crime = {df[LABEL_COLUMN].mean():.1%}, "
          f"country={meta['country']}, doc_type={meta['doc_type']}")
    return df


print("Loading CAP data for 7 sub-populations...\n")

subpops = {}
for key in DATA_PATHS:
    subpops[key] = load_cap_subpop(key)

print(f"\nTotal documents: {sum(len(df) for df in subpops.values()):,}")

In [ ]:
# --- Join LLM inference scores ---
# This step runs after the LLM inference pipeline has produced per-document
# score CSVs. Each inference output file has columns: id, score
# We join on 'id' and rename 'score' -> 'llm_score'.
#
# TODO: Update INFERENCE_DIR once the inference pipeline output is available.

INFERENCE_DIR = os.path.join('data', 'inference_output')

INFERENCE_PATHS = {
    'denmark_questions':   os.path.join(INFERENCE_DIR, 'denmark_questions_scores.csv'),
    'spain_questions':     os.path.join(INFERENCE_DIR, 'spain_questions_scores.csv'),
    'spain_elpais':        os.path.join(INFERENCE_DIR, 'spain_media_elpais_scores.csv'),
    'spain_elmundo':       os.path.join(INFERENCE_DIR, 'spain_media_elmundo_scores.csv'),
    'us_bills':            os.path.join(INFERENCE_DIR, 'us_bills_scores.csv'),
    'belgium_tv':          os.path.join(INFERENCE_DIR, 'belgium_tv_scores.csv'),
    'belgium_newspaper':   os.path.join(INFERENCE_DIR, 'belgium_newspaper_scores.csv'),
}

print("Joining LLM inference scores...\n")
for key, df in subpops.items():
    scores_df = pd.read_csv(INFERENCE_PATHS[key])
    # Inference output has 'score' column; rename to 'llm_score'
    scores_df = scores_df.rename(columns={'score': SCORE_COLUMN})
    n_before = len(df)
    subpops[key] = df.merge(scores_df[['id', SCORE_COLUMN]], on='id', how='inner')
    n_after = len(subpops[key])
    print(f"  {key}: {n_after:,}/{n_before:,} matched "
          f"(mean score = {subpops[key][SCORE_COLUMN].mean():.3f})")

print(f"\nTotal scored documents: {sum(len(df) for df in subpops.values()):,}")

In [ ]:
# Basic data exploration
print("=== Law & Crime prevalence by sub-population ===")
for key, df in subpops.items():
    meta = SUBPOP_META[key]
    print(f"  {meta['country']:>15} / {meta['doc_type']:<25}: "
          f"{df[LABEL_COLUMN].mean():.1%}  "
          f"({df[LABEL_COLUMN].sum():>5,} / {len(df):>7,})")

print("\n=== Year range by sub-population ===")
for key, df in subpops.items():
    print(f"  {key}: {df['year'].min()}--{df['year'].max()}")

print("\n=== Party coverage ===")
for key, df in subpops.items():
    if 'party' in df.columns and df['party'].notna().any():
        n_parties = df['party'].nunique()
        coverage = df['party'].notna().mean()
        print(f"  {key}: {n_parties} parties, {coverage:.0%} coverage")
    else:
        print(f"  {key}: no party data")

print("\n=== LLM score distribution ===")
for key, df in subpops.items():
    print(f"  {key}: mean={df[SCORE_COLUMN].mean():.3f}, "
          f"std={df[SCORE_COLUMN].std():.3f}")

## 2. Calibration Setup

Calibrate on a **balanced sample from 3 sub-populations** to give MCGrad
variation in doc_type, country, language, decade, and party:
- Denmark questions (~15K)
- Spain questions (~15K)
- US bills (~15K)

Total calibration: ~45K. The remaining data from these 3 sub-populations
forms the in-distribution test set.

**OOD targets** (completely unseen during calibration):
- Spain media (El Pais + El Mundo) -- same country, different doc type
- Belgium TV news -- new country, new language, new doc type
- Belgium newspaper -- new country, new language, different doc type

In [ ]:
# Build calibration set: balanced sample from 3 sub-populations
CAL_SUBPOPS = ['denmark_questions', 'spain_questions', 'us_bills']
CAL_N_PER_SUBPOP = 15_000

calibration_parts = []
test_parts = []

for key in CAL_SUBPOPS:
    df = subpops[key].copy()
    # Stratified split: take CAL_N_PER_SUBPOP for calibration, rest for test
    cal_frac = min(CAL_N_PER_SUBPOP / len(df), 0.40)
    cal_part, test_part = train_test_split(
        df,
        test_size=1.0 - cal_frac,
        random_state=42,
        stratify=df[LABEL_COLUMN],
    )
    calibration_parts.append(cal_part)
    test_parts.append(test_part)
    print(f"  {key}: {len(cal_part):,} calibration, {len(test_part):,} test")

calibration_df = pd.concat(calibration_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print(f"\nCalibration set: {len(calibration_df):,} samples")
print(f"  Law & Crime prevalence: {calibration_df[LABEL_COLUMN].mean():.1%}")
print(f"  By sub-pop: {calibration_df['subpop'].value_counts().to_dict()}")
print(f"  By country: {calibration_df['country'].value_counts().to_dict()}")
print(f"  By doc_type: {calibration_df['doc_type'].value_counts().to_dict()}")

print(f"\nIn-distribution test set: {len(test_df):,} samples")
print(f"  Law & Crime prevalence: {test_df[LABEL_COLUMN].mean():.1%}")

In [ ]:
# --- Feature preparation ---
# Create decade feature and fill missing party for all DataFrames

all_dfs_to_prep = [calibration_df, test_df] + [
    subpops[k] for k in subpops if k not in CAL_SUBPOPS
]
for df in all_dfs_to_prep:
    df['decade'] = (df['year'] // 10) * 10
    # Fill missing party with 'unknown' for MCGrad compatibility
    if 'party' in df.columns:
        df['party'] = df['party'].fillna('unknown')
    else:
        df['party'] = 'unknown'

In [ ]:
# --- Fit calibration methods ---

# 1. Isotonic Regression (global calibration)
isotonic_reg = mcgrad_methods.IsotonicRegression().fit(
    calibration_df,
    SCORE_COLUMN,
    LABEL_COLUMN,
)
print("Isotonic regression fitted")

# 2. MCGrad (multicalibration)
# Segment features: doc_type, country, party (categorical) + decade (numerical)
# The calibration set now has variation in all of these.
CATEGORICAL_SEGMENT_FEATURES = ['doc_type', 'country', 'party']
NUMERICAL_SEGMENT_FEATURES = ['decade']

mcgrad = mcgrad_methods.MCGrad()
mcgrad = mcgrad.fit(
    calibration_df,
    SCORE_COLUMN,
    LABEL_COLUMN,
    categorical_feature_column_names=CATEGORICAL_SEGMENT_FEATURES,
    numerical_feature_column_names=NUMERICAL_SEGMENT_FEATURES,
)
print("MCGrad fitted")

In [ ]:
# Generate calibrated predictions for test set and all OOD sub-populations
IR_COL = 'isotonic_prediction'
MCGRAD_COL = 'mcgrad_prediction'

# OOD sub-populations (not part of calibration split)
OOD_KEYS = [k for k in subpops if k not in CAL_SUBPOPS]

eval_dfs = [test_df] + [subpops[k] for k in OOD_KEYS]

for df in eval_dfs:
    # Isotonic regression predictions
    df[IR_COL] = isotonic_reg.predict(df, SCORE_COLUMN)

    # MCGrad predictions
    df[MCGRAD_COL] = mcgrad.predict(
        df=df,
        prediction_column_name=SCORE_COLUMN,
        categorical_feature_column_names=CATEGORICAL_SEGMENT_FEATURES,
        numerical_feature_column_names=NUMERICAL_SEGMENT_FEATURES,
    )

print("Calibrated predictions generated for all evaluation datasets")

In [ ]:
# --- Calibration parameters for quantification methods ---

def calibrate_threshold_prevalence_matching(labels, predictions):
    """Find threshold where apparent prevalence matches true prevalence."""
    true_prevalence = labels.mean()
    threshold = predictions.quantile(1 - true_prevalence)
    return float(threshold)


def estimate_classifier_error_rates(labels, predictions, threshold):
    """Estimate TPR and FPR from calibration data at given threshold."""
    binary_preds = (predictions >= threshold).astype(int)
    labels_arr = labels.astype(int)
    tp = ((binary_preds == 1) & (labels_arr == 1)).sum()
    fp = ((binary_preds == 1) & (labels_arr == 0)).sum()
    tn = ((binary_preds == 0) & (labels_arr == 0)).sum()
    fn = ((binary_preds == 0) & (labels_arr == 1)).sum()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return tpr, fpr


THRESHOLD = calibrate_threshold_prevalence_matching(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[SCORE_COLUMN],
)
print(f"Calibrated threshold (prevalence-matching): {THRESHOLD:.4f}")

# TPR and FPR at calibrated threshold for Rogan-Gladen
calibration_tpr, calibration_fpr = estimate_classifier_error_rates(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[SCORE_COLUMN],
    threshold=THRESHOLD,
)
print(f"\nCalibration set error rates at threshold={THRESHOLD:.4f}:")
print(f"  TPR (sensitivity): {calibration_tpr:.4f}")
print(f"  FPR (1-specificity): {calibration_fpr:.4f}")

# PACC parameters: E[h(X)|Y=1] and E[h(X)|Y=0]
PACC_POS_MEAN = calibration_df[calibration_df[LABEL_COLUMN] == 1][SCORE_COLUMN].mean()
PACC_NEG_MEAN = calibration_df[calibration_df[LABEL_COLUMN] == 0][SCORE_COLUMN].mean()
SOURCE_PREVALENCE = calibration_df[LABEL_COLUMN].mean()

print(f"\nPACC parameters (soft-score Rogan-Gladen):")
print(f"  E[h(X)|Y=1]: {PACC_POS_MEAN:.4f}")
print(f"  E[h(X)|Y=0]: {PACC_NEG_MEAN:.4f}")
print(f"\nSource prevalence for SLD: {SOURCE_PREVALENCE:.4f}")

# Verify: apparent prevalence matches true prevalence at calibrated threshold
apparent_prev = (calibration_df[SCORE_COLUMN] >= THRESHOLD).mean()
true_prev = calibration_df[LABEL_COLUMN].mean()
print(f"\nVerification:")
print(f"  True prevalence in calibration set: {true_prev:.4f}")
print(f"  Apparent prevalence at calibrated threshold: {apparent_prev:.4f}")

## 3. Shift Gradient Construction

We construct a gradient of increasing distributional shift:

| Scenario | Source | Shift type |
|----------|--------|------------|
| Baseline | Balanced test split (same composition as calibration) | None |
| Doc-type shift | Test split resampled to overweight US bills | Within-calibration |
| Party shift | Test split resampled to overweight one party | Within-calibration |
| OOD same language | Spain media (El Pais + El Mundo) | Cross-doc-type |
| OOD new country | Belgium TV news | New country + language + doc type |
| OOD new country | Belgium newspaper | New country + language + doc type |

In [ ]:
def resample_with_party_shift(
    df,
    shift='original',
    n_samples=20_000,
    random_state=42,
):
    """
    Resample df with importance weights that shift party composition.

    shift options:
      - 'original': uniform weights (baseline)
      - 'left_heavy': oversample parties with low law_crime prevalence rank
      - 'right_heavy': oversample parties with high law_crime prevalence rank
      - 'polarized': oversample both extremes, undersample middle

    Uses data-driven party ranking by law_crime topic prevalence.
    """
    actual_n = min(n_samples, len(df))
    if shift == 'original':
        weights = np.ones(len(df))
    else:
        party_topic_rate = df.groupby('party')[LABEL_COLUMN].mean()
        party_rank = party_topic_rate.rank(pct=True)
        rank_values = df['party'].map(party_rank).values

        if shift == 'left_heavy':
            weights = np.exp(-2.0 * rank_values)
        elif shift == 'right_heavy':
            weights = np.exp(2.0 * rank_values)
        elif shift == 'polarized':
            weights = np.exp(2.0 * np.abs(rank_values - 0.5))
        else:
            raise ValueError(f"Unknown shift: {shift}")

    weights = weights / weights.sum()
    return df.sample(
        n=actual_n,
        weights=weights,
        replace=True,
        random_state=random_state,
    )


def resample_with_doctype_shift(
    df,
    target_doctype='bill',
    overweight_factor=5.0,
    n_samples=20_000,
    random_state=42,
):
    """
    Resample df to overweight a specific doc_type.
    """
    actual_n = min(n_samples, len(df))
    weights = np.where(
        df['doc_type'] == target_doctype,
        overweight_factor,
        1.0,
    )
    weights = weights / weights.sum()
    return df.sample(
        n=actual_n,
        weights=weights,
        replace=True,
        random_state=random_state,
    )


# --- Build the shift gradient ---
N_SAMPLES = 20_000

# Combine Spain media sub-populations for OOD target
spain_media_df = pd.concat(
    [subpops['spain_elpais'], subpops['spain_elmundo']],
    ignore_index=True,
)

scenarios = {
    'Baseline\n(balanced test)': {
        'df': test_df.sample(
            n=min(N_SAMPLES, len(test_df)),
            replace=True, random_state=42),
        'shift_type': 'none',
    },
    'Doc-type shift\n(overweight bills)': {
        'df': resample_with_doctype_shift(
            test_df, target_doctype='bill',
            n_samples=N_SAMPLES, random_state=42),
        'shift_type': 'within-calibration',
    },
    'Party shift\n(right-heavy)': {
        'df': resample_with_party_shift(
            test_df, shift='right_heavy',
            n_samples=N_SAMPLES, random_state=42),
        'shift_type': 'within-calibration',
    },
    'Spain media\n(OOD doc type)': {
        'df': spain_media_df.sample(
            n=min(N_SAMPLES, len(spain_media_df)),
            replace=True, random_state=42),
        'shift_type': 'OOD same-language',
    },
    'Belgium TV\n(OOD country)': {
        'df': subpops['belgium_tv'].sample(
            n=min(N_SAMPLES, len(subpops['belgium_tv'])),
            replace=True, random_state=42),
        'shift_type': 'OOD new-country',
    },
    'Belgium newspaper\n(OOD country)': {
        'df': subpops['belgium_newspaper'].sample(
            n=min(N_SAMPLES, len(subpops['belgium_newspaper'])),
            replace=True, random_state=42),
        'shift_type': 'OOD new-country',
    },
}

print("Shift gradient scenarios:")
print(f"{'Scenario':<30} {'N':>7}  {'True prev':>10}  {'Shift type'}")
print("-" * 75)
for label, info in scenarios.items():
    df = info['df']
    clean_label = label.replace('\n', ' ')
    print(f"{clean_label:<30} {len(df):>7,}  "
          f"{df[LABEL_COLUMN].mean():>9.1%}  {info['shift_type']}")

## 4. Prevalence Estimation

For each scenario along the shift gradient, compute prevalence estimates
with all 7 methods:
1. **Raw LLM scores** (mean of uncalibrated scores)
2. **Classify & Count** (fraction above threshold)
3. **Rogan-Gladen** (CC adjusted by TPR/FPR)
4. **PACC** (soft-score Rogan-Gladen)
5. **SLD (EMQ)** (EM algorithm)
6. **Isotonic Regression** (mean of isotonic-calibrated scores)
7. **MCGrad** (mean of multicalibration-adjusted scores)

In [ ]:
def sld_estimate(scores, source_prevalence, max_iter=100, tol=1e-6):
    """Saerens-Latinne-Decaestecker (EMQ) prevalence estimator.

    EM algorithm that iteratively re-estimates prevalence by adjusting
    posteriors for a new prior. Assumes label shift (P(X|Y) stable).
    """
    p_hat = source_prevalence
    for _ in range(max_iter):
        ratio_pos = p_hat / source_prevalence
        ratio_neg = (1 - p_hat) / (1 - source_prevalence)
        adjusted = (ratio_pos * scores) / (ratio_pos * scores + ratio_neg * (1 - scores))
        p_new = adjusted.mean()
        if abs(p_new - p_hat) < tol:
            break
        p_hat = p_new
    return p_hat


def pacc_estimate(scores, pos_mean, neg_mean):
    """Probabilistic Adjusted Classify & Count.

    Soft-score generalization of Rogan-Gladen: uses E[h(X)|Y=1] and E[h(X)|Y=0]
    instead of binary TPR/FPR.
    """
    pcc = scores.mean()
    denom = pos_mean - neg_mean
    if abs(denom) < 1e-10:
        return pcc
    return np.clip((pcc - neg_mean) / denom, 0.0, 1.0)


def compute_rogan_gladen_estimate(apparent_prevalence, tpr, fpr):
    """Rogan-Gladen adjusted prevalence estimate."""
    denominator = tpr - fpr
    if abs(denominator) < 1e-10:
        return apparent_prevalence
    adjusted = (apparent_prevalence - fpr) / denominator
    return max(0.0, min(1.0, adjusted))


def compute_all_prevalence_estimates(
    target_df,
    score_col,
    ir_col,
    mcgrad_col,
    label_col,
    cal_tpr,
    cal_fpr,
    pacc_pos_mean,
    pacc_neg_mean,
    source_prevalence,
    threshold=0.5,
):
    """Compute prevalence estimates using all 7 methods."""
    true_prevalence = target_df[label_col].mean()

    # 1. Raw LLM scores
    raw_estimate = target_df[score_col].mean()

    # 2. Classify & Count
    binary_preds = (target_df[score_col] >= threshold).astype(int)
    classify_count = binary_preds.mean()

    # 3. Rogan-Gladen
    rogan_gladen = compute_rogan_gladen_estimate(classify_count, cal_tpr, cal_fpr)

    # 4. PACC
    pacc = pacc_estimate(target_df[score_col].values, pacc_pos_mean, pacc_neg_mean)

    # 5. SLD (EMQ)
    sld = sld_estimate(target_df[score_col].values, source_prevalence)

    # 6. Isotonic Regression
    isotonic_estimate = target_df[ir_col].mean()

    # 7. MCGrad
    mcgrad_estimate = target_df[mcgrad_col].mean()

    return {
        'True Prevalence': true_prevalence,
        'Raw Scores': raw_estimate,
        'Classify & Count': classify_count,
        'Rogan-Gladen': rogan_gladen,
        'PACC': pacc,
        'SLD (EMQ)': sld,
        'Isotonic Regression': isotonic_estimate,
        'MCGrad': mcgrad_estimate,
    }


def compute_bias_table(estimates):
    """Create a DataFrame showing estimates and bias for each method."""
    true_prev = estimates['True Prevalence']
    rows = []
    for method, estimate in estimates.items():
        if method == 'True Prevalence':
            continue
        bias = estimate - true_prev
        rows.append({
            'Method': method,
            'Estimate': estimate,
            'Bias': bias,
            'Relative Bias (%)': 100 * bias / true_prev if true_prev > 0 else 0,
        })
    return pd.DataFrame(rows).set_index('Method')


# Methods to compare (same as ACS)
methods_to_plot = [
    'Raw Scores', 'Classify & Count', 'Rogan-Gladen', 'PACC',
    'SLD (EMQ)', 'Isotonic Regression', 'MCGrad',
]

colors_map = {m: METHOD_COLORS[m] for m in methods_to_plot}

In [ ]:
# Compute prevalence estimates for each scenario along the shift gradient
all_results = {}

for scenario_label, info in scenarios.items():
    syn_df = info['df']
    estimates = compute_all_prevalence_estimates(
        target_df=syn_df,
        score_col=SCORE_COLUMN,
        ir_col=IR_COL,
        mcgrad_col=MCGRAD_COL,
        label_col=LABEL_COLUMN,
        cal_tpr=calibration_tpr,
        cal_fpr=calibration_fpr,
        pacc_pos_mean=PACC_POS_MEAN,
        pacc_neg_mean=PACC_NEG_MEAN,
        source_prevalence=SOURCE_PREVALENCE,
        threshold=THRESHOLD,
    )
    all_results[scenario_label] = estimates
    clean_label = scenario_label.replace('\n', ' ')
    print(f"\n{clean_label}:")
    print(f"  True prevalence: {estimates['True Prevalence']:.4f}")
    display(compute_bias_table(estimates).round(4))

## 5. Bootstrap RMSE

200 bootstrap iterations per scenario. For each iteration, resample with
replacement from the scenario's source population and compute prevalence
estimates. Report bias and RMSE across resamples.

Bootstrap sample size: `min(len(source_population), 20_000)` to handle
sub-populations smaller than 20K (e.g., Belgium newspaper ~21K).

In [ ]:
def compute_bootstrap_rmse(
    source_df,
    score_col,
    ir_col,
    mcgrad_col,
    label_col,
    cal_tpr,
    cal_fpr,
    pacc_pos_mean,
    pacc_neg_mean,
    source_prevalence,
    threshold,
    n_bootstrap=200,
    n_samples=20_000,
):
    """Bootstrap resampling to compute bias, variance, and RMSE.

    Resamples uniformly from the source population (no additional shift).
    Sample size is min(n_samples, len(source_df)).
    """
    actual_n = min(n_samples, len(source_df))
    methods_list = [
        'Raw Scores', 'Classify & Count', 'Rogan-Gladen', 'PACC',
        'SLD (EMQ)', 'Isotonic Regression', 'MCGrad',
    ]
    estimates_by_method = {m: [] for m in methods_list}
    true_prevs = []

    for b in range(n_bootstrap):
        syn_df = source_df.sample(
            n=actual_n,
            replace=True,
            random_state=b,
        )
        est = compute_all_prevalence_estimates(
            syn_df, score_col, ir_col, mcgrad_col, label_col,
            cal_tpr, cal_fpr, pacc_pos_mean, pacc_neg_mean,
            source_prevalence, threshold,
        )
        true_prevs.append(est['True Prevalence'])
        for m in methods_list:
            estimates_by_method[m].append(est[m])

    results = {}
    for m in methods_list:
        ests = np.array(estimates_by_method[m])
        trues = np.array(true_prevs)
        errors = ests - trues
        results[m] = {
            'bias': np.mean(errors) * 100,       # in percentage points
            'variance': np.var(errors) * 100**2,  # in pp^2
            'rmse': np.sqrt(np.mean(errors**2)) * 100,  # in pp
        }
    results['True Prevalence'] = np.mean(true_prevs)
    return results


def compute_bootstrap_rmse_shifted(
    source_df,
    shift_fn,
    shift_kwargs,
    score_col,
    ir_col,
    mcgrad_col,
    label_col,
    cal_tpr,
    cal_fpr,
    pacc_pos_mean,
    pacc_neg_mean,
    source_prevalence,
    threshold,
    n_bootstrap=200,
):
    """Bootstrap with a shift resampling function (for within-calibration scenarios)."""
    methods_list = [
        'Raw Scores', 'Classify & Count', 'Rogan-Gladen', 'PACC',
        'SLD (EMQ)', 'Isotonic Regression', 'MCGrad',
    ]
    estimates_by_method = {m: [] for m in methods_list}
    true_prevs = []

    for b in range(n_bootstrap):
        syn_df = shift_fn(source_df, random_state=b, **shift_kwargs)
        est = compute_all_prevalence_estimates(
            syn_df, score_col, ir_col, mcgrad_col, label_col,
            cal_tpr, cal_fpr, pacc_pos_mean, pacc_neg_mean,
            source_prevalence, threshold,
        )
        true_prevs.append(est['True Prevalence'])
        for m in methods_list:
            estimates_by_method[m].append(est[m])

    results = {}
    for m in methods_list:
        ests = np.array(estimates_by_method[m])
        trues = np.array(true_prevs)
        errors = ests - trues
        results[m] = {
            'bias': np.mean(errors) * 100,
            'variance': np.var(errors) * 100**2,
            'rmse': np.sqrt(np.mean(errors**2)) * 100,
        }
    results['True Prevalence'] = np.mean(true_prevs)
    return results

In [ ]:
print("Computing bootstrap RMSE (200 resamples per scenario)...")

# Bootstrap source configuration for each scenario.
# 'mode': 'uniform' = simple resampling, 'shift' = apply shift function
bootstrap_config = {
    'Baseline\n(balanced test)': {
        'mode': 'uniform', 'source': test_df,
    },
    'Doc-type shift\n(overweight bills)': {
        'mode': 'shift', 'source': test_df,
        'shift_fn': resample_with_doctype_shift,
        'shift_kwargs': {'target_doctype': 'bill', 'n_samples': N_SAMPLES},
    },
    'Party shift\n(right-heavy)': {
        'mode': 'shift', 'source': test_df,
        'shift_fn': resample_with_party_shift,
        'shift_kwargs': {'shift': 'right_heavy', 'n_samples': N_SAMPLES},
    },
    'Spain media\n(OOD doc type)': {
        'mode': 'uniform', 'source': spain_media_df,
    },
    'Belgium TV\n(OOD country)': {
        'mode': 'uniform', 'source': subpops['belgium_tv'],
    },
    'Belgium newspaper\n(OOD country)': {
        'mode': 'uniform', 'source': subpops['belgium_newspaper'],
    },
}

common_args = dict(
    score_col=SCORE_COLUMN, ir_col=IR_COL, mcgrad_col=MCGRAD_COL,
    label_col=LABEL_COLUMN, cal_tpr=calibration_tpr, cal_fpr=calibration_fpr,
    pacc_pos_mean=PACC_POS_MEAN, pacc_neg_mean=PACC_NEG_MEAN,
    source_prevalence=SOURCE_PREVALENCE, threshold=THRESHOLD,
)

bootstrap_results = {}
for label, cfg in bootstrap_config.items():
    clean_label = label.replace('\n', ' ')
    print(f"  {clean_label}...")
    if cfg['mode'] == 'shift':
        bootstrap_results[label] = compute_bootstrap_rmse_shifted(
            cfg['source'],
            shift_fn=cfg['shift_fn'],
            shift_kwargs=cfg['shift_kwargs'],
            **common_args,
        )
    else:
        bootstrap_results[label] = compute_bootstrap_rmse(
            cfg['source'], **common_args,
        )

print("Done.")

## 6. Results Table

Summary table showing bias and RMSE across the shift gradient,
mirroring the ACS Table 1 format. Rows ordered from no shift to maximum shift.

In [ ]:
# Summary table: bias and RMSE across shift gradient
shift_summary = []

scenario_order = list(scenarios.keys())

for scenario_label in scenario_order:
    estimates = all_results[scenario_label]
    bs = bootstrap_results[scenario_label]
    true_prev = estimates['True Prevalence']
    clean_label = scenario_label.replace('\n', ' ')
    shift_type = scenarios[scenario_label]['shift_type']
    row = {
        'Scenario': clean_label,
        'Shift Type': shift_type,
        'True Prevalence': f'{true_prev:.1%}',
    }
    for method in methods_to_plot:
        bias_pp = (estimates[method] - true_prev) * 100
        rmse_pp = bs[method]['rmse']
        row[f'{method} Bias'] = f'{bias_pp:+.2f}pp'
        row[f'{method} RMSE'] = f'{rmse_pp:.2f}pp'
    shift_summary.append(row)

summary_df = pd.DataFrame(shift_summary).set_index(['Scenario', 'Shift Type'])
print('=== Law & Crime Prevalence Estimation: Bias and RMSE Across Shift Gradient ===\n')
summary_df

## 7. Figure: Bias Across the Shift Gradient

Single-panel bar chart showing prevalence estimation bias for each method
across the shift gradient (from baseline to maximum shift).
Uses `METHOD_COLORS` from `plot_config.py`, same style as ACS Figure 2.

In [ ]:
scenario_labels = list(scenarios.keys())

fig, ax = plt.subplots(figsize=(14, 6.5))

x = np.arange(len(scenario_labels))
n_methods = len(methods_to_plot)
group_width = 0.82
bar_width = group_width / n_methods * 0.85

for i, method in enumerate(methods_to_plot):
    biases = [
        (all_results[s][method] - all_results[s]['True Prevalence']) * 100
        for s in scenario_labels
    ]
    offset = (i - n_methods / 2 + 0.5) * (group_width / n_methods)
    ax.bar(
        x + offset, biases, bar_width,
        label=method, color=colors_map[method],
        edgecolor='white', linewidth=0.5,
    )

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Bias (percentage points)')
ax.set_xticks(x)
ax.set_xticklabels(scenario_labels, rotation=45, ha='right')
ax.legend(fontsize=7, loc='best')

# Add shift gradient annotation
ax.annotate(
    '', xy=(len(scenario_labels) - 0.5, ax.get_ylim()[0]),
    xytext=(-0.5, ax.get_ylim()[0]),
    arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5),
)
ax.text(
    len(scenario_labels) / 2 - 0.5, ax.get_ylim()[0] * 0.95,
    'increasing distributional shift $\\longrightarrow$',
    ha='center', va='top', fontsize=9, color='#888888', style='italic',
)

fig.suptitle(
    'Law & Crime Prevalence Estimation Bias Across Shift Gradient',
    fontsize=13,
)
fig.tight_layout()
fig.savefig(
    '../paper/images/figure_cap_shift_gradient.png',
    dpi=300, bbox_inches='tight',
)
plt.show()